## How to use this notebook

Run this first, top to bottom, from the repository root (or from `notebooks/`). It downloads the
raw dataset, cleans it, removes near-duplicates, and writes `data/processed/cleaned_recipes.parquet`.
Shared helpers now live in `src/preprocessing.py`; see its docstring for a shorter programmatic path.

# 01 - Preprocessing: FILIPINO_RECIPES_2K
Outputs: `data/processed/cleaned_recipes.parquet` (and later `alias_map.json`).

Run the cells in order. Cell 7 needs a manual decision before you continue.

In [17]:
import sys, os
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
os.chdir(ROOT)
sys.path.insert(0, str(ROOT / "src"))
print("Working directory:", Path.cwd())

Working directory: c:\Users\micheala\Desktop\pinoy-recipe-rag\NLP-Mini-Project


## 0. Install (run once, then restart the kernel)

In [18]:
import sys
print(sys.executable)
%pip install datasets pyarrow pandas rapidfuzz

c:\Users\micheala\Desktop\pinoy-recipe-rag\.venv\Scripts\python.exe


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 23.2.1 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


## 1. Load the dataset and save the raw copy

In [19]:
from datasets import load_dataset
import os

ds = load_dataset("joackimagno/FILIPINO_RECIPES_2K")
print(ds)

split = list(ds.keys())[0]
df = ds[split].to_pandas()

os.makedirs("data/raw", exist_ok=True)
os.makedirs("data/processed", exist_ok=True)
df.to_parquet("data/raw/raw_recipes.parquet", index=False)
print(df.shape)

DatasetDict({
    train: Dataset({
        features: ['recipe_name', 'prep_time', 'cook_time', 'total_time', 'servings', 'ingredients', 'instructions', 'ingredient_names'],
        num_rows: 1988
    })
})
(1988, 8)


## 2. Inspect structure and problems

In [20]:
import pandas as pd
print(df.dtypes)
print("\nNulls:\n", df.isna().sum())
print("\nEmpty strings:\n", (df.astype(str).apply(lambda c: c.str.strip() == "")).sum())
print("\nExact duplicate rows:", df.duplicated().sum())
print("Duplicate recipe names:", df["recipe_name"].duplicated().sum())

recipe_name         str
prep_time           str
cook_time           str
total_time          str
servings            str
ingredients         str
instructions        str
ingredient_names    str
dtype: object

Nulls:
 recipe_name         0
prep_time           0
cook_time           0
total_time          0
servings            0
ingredients         0
instructions        0
ingredient_names    0
dtype: int64

Empty strings:
 recipe_name         0
prep_time           0
cook_time           0
total_time          0
servings            0
ingredients         0
instructions        0
ingredient_names    0
dtype: int64

Exact duplicate rows: 0
Duplicate recipe names: 98


## 3. Read actual rows

In [21]:
df.sample(3, random_state=1).T

,1549,1546,1179
recipe_name,Sarciadong Manok Recipe,Adobong Pula (Red Adobo) Recipe,Meaty Spaghetti Recipe
prep_time,10 minutes,5 minutes,10 minutes
cook_time,20 minutes,1 hour,30 minutes
total_time,30 minutes,1 hour 5 minutes,40 minutes
servings,3 to 4,4 to 6,4
ingredients,500 Grams Chicken (use different parts) | 2 ta...,3 tablespoons cooking oil | 1/4 Cup Achuete Se...,2 tablespoons cooking oil | 3 cloves Garlic (p...
instructions,Heat oil in a pot over medium heat. Saute onio...,Heat cooking oil in a saucepan over medium hea...,"In a pot, saute garlic in cooking oil until br..."
ingredient_names,different part | canola oil | laurel leaf | to...,achuete seed | pork | dried bay leaf,pork | beef | sweet style spaghetti sauce | ca...


## 4. Look at the ingredient fields closely

In [22]:
for i in range(3):
    print(repr(df.loc[i, "ingredients"]))
    print(repr(df.loc[i, "ingredient_names"]))
    print(type(df.loc[i, "ingredient_names"]))
    print()

'3 lbs. pork hocks cleaned | 1 1/2 cups mung beans washed | 1 ounce sotanghon | 2 teaspoons beef powder | 1 onion chopped | 5 garlic chopped | 3 tomatoes diced | 2 cups baby spinach | 8 cups water | Fish sauce and ground black pepper to taste | 3 tablespoons cooking oil'
'pork | mung bean | sotanghon | beef powder | tomato | baby spinach'
<class 'str'>

'3 lbs. kalabasa cubed | 18 pieces string beans cut into 2-inch pieces | 1 1/2 lbs. pork belly diced | 24 ounces coconut milk | 2 tablespoon shrimp paste | 1 onion chopped | 3 thumbs ginger julienned | 8 cloves garlic | 8 grams Maggi Magic Sarap | 1 cup water'
'kalabasa | sitaw | pork | coconut milk | thumbs ginger'
<class 'str'>

'2 lbs. bangus milkfish cleaned and sliced | 25 g Maggi Magic Sinigang sa Sampaloc | 1 bunch fresh kangkong leaves | 12 string beans cut into 2 inch pieces | 10 okra | 4 pieces long green pepper | 2 tomatoes wedged | 2 tablespoons miso | 1 yellow onion wedged | Fish sauce and ground black pepper to taste | 2 t

## 5. (Optional) Look at recipes that share a name

In [23]:
dup_names = df[df["recipe_name"].duplicated(keep=False)].sort_values("recipe_name")
print(len(dup_names))
dup_names[["recipe_name", "total_time", "servings", "ingredients"]].head(20)

185


,recipe_name,total_time,servings,ingredients
527,Adobo Burrito Recipe,30 minutes,4.0,1 tablespoon chopped garlic | 3 cups cooked ri...
1003,Adobo Burrito Recipe,30 minutes,4,1 tablespoon garlic (chopped) | 3 cups rice (c...
349,Adobo Pork Chops,1 hour 10 minutes,4.0,2 lbs. pork chops | 4 tablespoons dark soy sau...
540,Adobo Pork Chops,1 hours 10 minutes,4.0,2 lbs pork chops | 4 tablespoons dark soy sauc...
770,Adobong Kangkong Recipe,25 mins,5.0,1 Bundle of Kangkong (cleaned and hard stems r...
1503,Adobong Kangkong Recipe,20 minutes,4,1 Small white onion (thinly sliced) | 6 to 7 c...
762,Adobong Pusit Recipe (Squid Adobo),45 mins,6.0,1 1/2 lbs squid (pusit) | 1/2 cup vinegar | 1 ...
1742,Adobong Pusit Recipe (Squid Adobo),30 minutes,4,3 tablespoons cooking oil | 4 cloves Garlic (m...
180,Adobong Pusit sa Gata,35 minutes,4.0,1 lb. medium sized squid cleaned | 2 cups coco...
864,Adobong Pusit sa Gata,40 minutes,4.0,"2 pounds squid|4 cloves garlic, peeled and cho..."


## 6. Clean and structure the data

This cell now calls `src/preprocessing.py`. The old inline cleaning code was removed; the module
is unit-tested and handles unicode, time parsing, and pipe-separated fields in one place.

In [ ]:
# Maintained cleaning lives in src/preprocessing.py (tested, pure functions).
from preprocessing import prepare_recipes, find_near_duplicate_pairs

clean, report = prepare_recipes(df)
pairs = find_near_duplicate_pairs(clean)
print("dropped for missing content:", report["dropped_empty"])
print("rows now:", len(clean))
print(len(pairs), "same-name pairs compared")

## 7a. Find candidate near-duplicate pairs (only among recipes with the same name)

`find_near_duplicate_pairs()` in `src/preprocessing.py` does this now.

In [ ]:
pairs.sort_values("ing_jaccard", ascending=False).head(40)

## 7b. Two tiers: auto-flag the clear copies, review the uncertain ones by eye

In [26]:
tier1 = pairs[(pairs.ing_jaccard >= 0.8) & (pairs.time_diff <= 10)]
print(len(tier1), "pairs auto-flagged (tier 1)")

tier2 = pairs[(pairs.ing_jaccard < 0.8) & (pairs.ing_jaccard >= 0.4) &
              (pairs.instr_sim >= 0.75) & (pairs.time_diff <= 20)]
print(len(tier2), "pairs to review (tier 2)\n")

for _, r in tier2.iterrows():
    print(f"=== {r['name']} | rows {r['i']} vs {r['j']} | jac {r['ing_jaccard']} instr {r['instr_sim']} time_diff {r['time_diff']}")
    print("A:", clean.loc[r["i"], "ingredients"][:250])
    print("B:", clean.loc[r["j"], "ingredients"][:250], "\n")

4 pairs auto-flagged (tier 1)
5 pairs to review (tier 2)

=== adobo burrito | rows 527 vs 1003 | jac 0.46 instr 0.8 time_diff 0
A: 1 tablespoon chopped garlic | 3 cups cooked rice | 1/2 cup leftover adobo sauce | 4 large flour tortillas heated on a dry pan | 1 cup shredded adobo flakes from leftover adobo | 1 piece salted egg itlog na maalat, finely chopped | 1/2 cup chopped tom
B: 1 tablespoon garlic (chopped) | 3 cups rice (cooked) | 1/2 cup leftover adobo sauce | 4 large flour burritos (heated on a dry pan) | 1 cup adobo flakes (shredded) | 1 cup salted egg salsa | 1 piece Salted Egg (itlog na maalat) (finely chopped) | 1/2  

=== dinuguan | rows 38 vs 727 | jac 0.71 instr 0.75 time_diff 0
A: 1 lb pork cut into cubes | 1 cup vinegar | 2 pieces long green pepper | 1 1/2 tablespoon brown sugar | 1 piece onion chopped finely | 1 tablespoon garlic minced | 3 tablespoon cooking oil | 1 cup water | 10 oz pork blood | 1 piece pork cube boullion 
B: 1 lb pork belly cut into cubes | 1 cup vi

## 7c. YOUR DECISION: list the tier 2 pairs that are true duplicates
Read the printout above. If A and B are the same ingredients in different wording, add `(i, j)` here. If they have different ingredients or amounts, they are real variants, so leave them out. When unsure, keep both.

In [27]:
confirmed = [(527, 1003)]   # edit this list

## 7d. Group copies into clusters (union-find)

`duplicate_clusters()` handles this; you still decide which tier-2 pairs to confirm.

In [ ]:
# Union-find clustering is also in src/preprocessing.py; confirmed pairs stay manual.
from collections import Counter
from preprocessing import duplicate_clusters

clusters, to_drop, review = duplicate_clusters(pairs, clean, confirmed_pairs=confirmed)
print(len(clusters), "clusters,", sum(len(x) for x in clusters), "rows involved")
print("review-only pairs not merged:", len(review))

## 7e. Check clusters of size 3+ for wrongly chained recipes
A high worst-pair Jaccard (about 0.7+) means the whole cluster really is one recipe. A low one (below about 0.6) means it was chained through a middle row: put one of its row ids in `exclude_rows` to keep that whole cluster.

In [ ]:
from preprocessing import ingredient_tokens

def jac(a, b):
    A = ingredient_tokens(clean.loc[a, "ingredient_lines"])
    B = ingredient_tokens(clean.loc[b, "ingredient_lines"])
    return len(A & B) / max(1, len(A | B))

for cl in clusters:
    if len(cl) >= 3:
        worst = min(jac(a, b) for a, b in __import__("itertools").combinations(cl, 2))
        print(f"{clean.loc[cl[0], 'recipe_name'][:40]:40s} rows {cl}  worst pair jaccard {worst:.2f}")

In [30]:
exclude_rows = []   # any cluster containing one of these row ids is left alone

## 7f. Drop all but one row per cluster (keep the one with the longest instructions)

In [ ]:
# Honour exclusions decided in the cell above, then drop one row per cluster.
if exclude_rows:
    cancelled = {r for cl in clusters if any(x in exclude_rows for x in cl) for r in cl}
    to_drop = [r for r in to_drop if r not in cancelled]

n0 = len(clean)
clean = clean.drop(index=to_drop)
print("removed near-duplicates:", n0 - len(clean))
print("rows now:", len(clean))
print("same name, different content (kept):", clean["recipe_name"].str.lower().duplicated().sum())

## 8. Final checks and save `cleaned_recipes.parquet`

In [32]:
clean = clean.reset_index(drop=True)
clean.insert(0, "recipe_id", range(len(clean)))
print(clean.shape)
print(clean[["total_time", "total_time_min", "servings", "servings_min", "servings_max"]].sample(min(8, len(clean)), random_state=1))
print(clean["total_time_min"].isna().sum(), "rows with unparsed total time")

clean.to_parquet("data/processed/cleaned_recipes.parquet", index=False)
pd.read_parquet("data/processed/cleaned_recipes.parquet").dtypes

(1983, 16)
             total_time  total_time_min servings  servings_min  servings_max
820          45 minutes              45      4.0             4             4
1227         55 minutes              55        6             6             6
1419         30 minutes              30        4             4             4
111   1 hour 10 minutes              70      4.0             4             4
480          50 minutes              50      5.0             5             5
1519         55 minutes              55        2             2             2
1546         40 minutes              40        4             4             4
56           50 minutes              50      5.0             5             5
0 rows with unparsed total time


recipe_id             int64
recipe_name             str
prep_time               str
cook_time               str
total_time              str
servings                str
ingredients             str
instructions            str
ingredient_names        str
ingredient_lines     object
ingredient_list      object
prep_time_min       float64
cook_time_min       float64
total_time_min        int64
servings_min          int64
servings_max          int64
dtype: object

## 9. Ingredient vocabulary (starting point for the alias map)

In [33]:
vocab = Counter(i for l in clean["ingredient_list"] for i in l)
print(len(vocab), "distinct ingredient names")
for term, n in vocab.most_common(80):
    print(f"{n:4d}  {term}")

2789 distinct ingredient names
 487  pork
 269  salt and pepper
 266  tomato
 247  carrot
 243  ginger
 240  chicken
 228  canola oil
 209  vegetable oil
 191  bell pepper
 185  brown sugar
 181  potato
 181  egg
 158  green onion
 147  shrimp
 139  coconut milk
 120  tomato sauce
 118  sitaw
 111  fish sauce fish sauce
 109  cabbage
 107  calamansi juice
 100  peppercorn
  99  calamansi
  94  beef
  92  onion sibuya
  91  cornstarch
  89  butter
  85  dried bay leave
  85  garlic bawang
  82  thai chili pepper
  80  okra
  78  salt asin
  75  whole peppercorn
  74  kangkong
  73  eggplant
  71  bay leaf
  70  water tubig
  69  all purpose flour
  69  coconut cream
  67  oyster sauce
  63  finger chili
  61  bay leave
  61  garlic powder
  59  olive oil
  57  sesame oil
  55  beef broth
  53  soy sauce soy sauce
  48  whole chicken
  46  green finger chili
  45  labuyo
  45  oil mantika
  44  chicken broth
  44  dahon ng laurel
  44  chicken stock
  44  black peppercorn
  43  malunggay